# 🔄 Convert PNG Dataset to NPY (For Cluster Transfer)

Ce notebook convertit vos images PNG multi-fenêtres en fichiers `.npy` et les compresse dans un fichier `.zip` unique. Cela résout le problème du transfert lent de milliers de petits fichiers vers le cluster.

Il met également à jour le fichier CSV des labels pour qu'il pointe vers les fichiers `.npy`.

In [ ]:
from google.colab import drive
import os
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import shutil

# 1. Monter Google Drive
drive.mount('/content/drive')

In [ ]:
# 2. Configuration des chemins
DRIVE_FOLDER = '/content/drive/MyDrive/artishow'
PNG_FOLDER = os.path.join(DRIVE_FOLDER, 'preprocessed_images_multiwindow')
NPY_FOLDER = os.path.join(DRIVE_FOLDER, 'preprocessed_images_npy')

CSV_INPUT = os.path.join(DRIVE_FOLDER, 'dataset_analysis_colab', 'dataset_labeled_with_png_mapping.csv')
CSV_OUTPUT = os.path.join(DRIVE_FOLDER, 'dataset_analysis_colab', 'dataset_labeled_with_npy_mapping.csv')

os.makedirs(NPY_FOLDER, exist_ok=True)
print(f'Dossier source PNG : {PNG_FOLDER}')
print(f'Dossier cible NPY : {NPY_FOLDER}')

In [ ]:
# 3. Conversion des PNG en NPY
png_files = [f for f in os.listdir(PNG_FOLDER) if f.endswith('.png')]
print(f'Trouvé {len(png_files)} images à convertir.')

for png_file in tqdm(png_files, desc='Conversion en .npy'):
    png_path = os.path.join(PNG_FOLDER, png_file)
    npy_filename = png_file.replace('.png', '.npy')
    npy_path = os.path.join(NPY_FOLDER, npy_filename)
    
    # Vérifier si déjà converti (reprise possible)
    if os.path.exists(npy_path):
        continue
        
    try:
        img = Image.open(png_path).convert('RGB')
        img_array = np.array(img, dtype=np.uint8)
        np.save(npy_path, img_array)
    except Exception as e:
        print(f'Erreur avec {png_file} : {e}')

print('✅ Conversion terminée !')

In [ ]:
# 4. Mise à jour du CSV des labels
print('Mise à jour du CSV des labels...')
df = pd.read_csv(CSV_INPUT)

# Remplacer l'extension .png par .npy dans la colonne png_filename ou créer npy_filename
df['npy_filename'] = df['png_filename'].apply(lambda x: str(x).replace('.png', '.npy'))

# Sauvegarder le nouveau CSV
df.to_csv(CSV_OUTPUT, index=False)
print(f'✅ Nouveau CSV sauvegardé à : {CSV_OUTPUT}')

In [ ]:
# 5. Création d'une archive ZIP pour faciliter le transfert vers le cluster Télécom
ZIP_PATH = os.path.join(DRIVE_FOLDER, 'dataset_npy_archive.zip')

print('Création de l\'archive ZIP (cela peut prendre quelques minutes)...')
!zip -r -q -0 {ZIP_PATH} {NPY_FOLDER}
print(f'✅ Archive ZIP créée à : {ZIP_PATH}')
print('Vous pouvez maintenant télécharger ce seul fichier ZIP depuis votre Google Drive, puis l\'envoyer (scp) sur le cluster et le dézipper (unzip).')